# Generation script

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.qasm2 import dumps
import numpy as np
import os
import time
import datetime
import json
from collections import Counter
from qiskit.transpiler.passes import ResourceEstimation
from IPython.display import display
import platform
import psutil

def get_system_info():
    info = {
        "platform": f"{platform.system()} {platform.release()}",
        "architecture": platform.machine(),
        "processor": platform.processor(),
        "cpu_count": os.cpu_count(),
        "total_ram_gb": round(psutil.virtual_memory().total / (1024**3), 2),
        "available_ram_gb": round(psutil.virtual_memory().available / (1024**3), 2)
    }
    try:
        import cpuinfo
        cpu = cpuinfo.get_cpu_info()
        info["cpu_brand"] = cpu.get('brand_raw', 'N/A')
        info["cpu_hz"] = cpu.get('hz_advertised_friendly', 'N/A')
    except ImportError:
        info["cpu_brand"] = 'N/A'
        info["cpu_hz"] = 'N/A'
    return info

def generate_random_clifford_t_circuit(num_qubits, h_count, t_count, s_count=0, z_count=0, cz_count=0):
    """Generate a random quantum circuit using {H, Z, S, T, CZ} gates."""
    qc = QuantumCircuit(num_qubits)
    qc.h(0)
    for i in range(num_qubits - 1):
        qc.cz(i, i + 1)

    # Add random H, S, Z gates
    for _ in range(h_count):
        qc.h(np.random.randint(0, num_qubits))
    for _ in range(s_count):
        qc.s(np.random.randint(0, num_qubits))
    for _ in range(z_count):
        qc.z(np.random.randint(0, num_qubits))

    # Add exactly t_count T gates, distributed randomly
    t_qubits = np.random.choice(num_qubits, size=t_count, replace=True)
    for qubit in t_qubits:
        qc.t(qubit)

    # Add random CZ gates
    for _ in range(cz_count):
        control = np.random.randint(0, num_qubits)
        target = np.random.randint(0, num_qubits)
        while target == control:
            target = np.random.randint(0, num_qubits)
        qc.cz(control, target)

    return qc

# Set fixed RNG seed for reproducibility
np.random.seed(42)

# Parameters for random circuits
num_qubits = 29
h_count = 10
s_count = 10
z_count = 25
cz_count = 10

t_counts = [4, 3, 2, 1]
test_dir = "test"
os.makedirs(test_dir, exist_ok=True)

simulator = AerSimulator(method='statevector')

for t_count in t_counts:
    random_qc = generate_random_clifford_t_circuit(num_qubits, h_count, t_count, s_count, z_count, cz_count)
    filename_base = f"test_q{num_qubits}_h{h_count}_t{t_count}_s{s_count}_z{z_count}_cz{cz_count}"

    # Export to QASM2
    qasm_filename = os.path.join(test_dir, f"{filename_base}.qasm")
    with open(qasm_filename, 'w') as f:
        f.write(dumps(random_qc))
    print(f"QASM saved to: {qasm_filename}")

    # Simulate and get statevector, benchmark simulation time
    random_qc.save_statevector()
    start_sim = time.time()
    result = simulator.run(random_qc).result()
    elapsed_sim = time.time() - start_sim
    statevector = result.get_statevector()
    print(f"\nQiskit simulation time (h={h_count}, t={t_count}): {elapsed_sim:.4f} seconds")
    print(f"First element of statevector: {statevector[0]}")

    # Save simulation time and system info as JSON
    siminfo = {
        "simulation_time_seconds": elapsed_sim,
        "system_info": get_system_info(),
        "date_time": datetime.datetime.now().isoformat()
    }
    siminfo_filename = os.path.join(test_dir, f"{filename_base}_siminfo.json")
    with open(siminfo_filename, 'w') as f:
        json.dump(siminfo, f, indent=2)
    print(f"Simulation info saved as: {siminfo_filename}")

    # Print gate counts
    gate_counts = Counter(random_qc.count_ops())
    print("Gate counts:")
    for gate in ['h', 't', 's', 'z', 'cz']:
        print(f"{gate.upper()}: {gate_counts.get(gate, 0)}")
    print('-' * 60)


QASM saved to: test/test.qasm

Qiskit simulation time: 42.9703 seconds
Simulation info saved as: test/test_siminfo.json
Gate counts:
H: 11
T: 5
S: 10
Z: 25
CZ: 38


In [1]:


# Generate and save additional random circuits for t counts 4, 3, 2, 1
output_base_dir = "test_random_t_variants"
os.makedirs(output_base_dir, exist_ok=True)

# Use constant H count for the new circuits and test reduced T gates
t_counts = [5, 3, 2, 1]
h_count_var = 10

for t_count_var in t_counts:
    random_qc_var = generate_random_clifford_t_circuit(num_qubits, h_count_var, t_count_var, s_count, z_count, cz_count)
    filename_base_var = f"random_circuit_q{num_qubits}_h{h_count_var}_t{t_count_var}_s{s_count}_z{z_count}_cz{cz_count}"
    circuit_dir = os.path.join(output_base_dir, filename_base_var)
    os.makedirs(circuit_dir, exist_ok=True)

    # Export to QASM2
    qasm_output_var = dumps(random_qc_var)
    qasm_filename = f"{circuit_dir}/{filename_base_var}.qasm"
    with open(qasm_filename, 'w') as f:
        f.write(qasm_output_var)
    print(f"QASM saved to: {qasm_filename}")

    # Simulate and get statevector, benchmark simulation time
    random_qc_var.save_statevector()
    start_sim_var = time.time()
    result_var = simulator.run(random_qc_var).result()
    elapsed_sim_var = time.time() - start_sim_var
    statevector_var = result_var.get_statevector()
    print(f"\nQiskit simulation time (h={h_count_var}, t={t_count_var}): {elapsed_sim_var:.4f} seconds")
    print(f"First element of statevector: {statevector_var[0]}")

    # Save simulation metadata
    siminfo_var = {
        "simulation_time_seconds": elapsed_sim_var,
        "system_info": get_system_info(),
        "date_time": datetime.datetime.now().isoformat()
    }
    siminfo_filename = f"{circuit_dir}/{filename_base_var}_siminfo.json"
    with open(siminfo_filename, 'w') as f:
        json.dump(siminfo_var, f, indent=2)
    print(f"Simulation info saved as: {siminfo_filename}")

    # Print gate counts for the new circuit
    gate_counts_var = Counter(random_qc_var.count_ops())
    print(f"Gate counts (h={h_count_var}, t={t_count_var}):")
    for gate in ['h', 't', 's', 'z', 'cz']:
        print(f"{gate.upper()}: {gate_counts_var.get(gate, 0)}")

NameError: name 'os' is not defined

# clean up script

In [2]:
import shutil
import os

def clean_dataset_folders():
    # Remove all contents of 'dataset' and '../dataset' folders
    for folder in ['dataset', '../dataset']:
        if os.path.exists(folder):
            for entry in os.listdir(folder):
                entry_path = os.path.join(folder, entry)
                if os.path.isdir(entry_path):
                    shutil.rmtree(entry_path)
                else:
                    os.remove(entry_path)
    print("All contents of 'dataset' and '../dataset' have been deleted.")

clean_dataset_folders()

All contents of 'dataset' and '../dataset' have been deleted.
